# FUNCIONES de Pyspark
- explode: extare un array de una columna
- coalesce: combina varias columnas en una 

In [5]:
import findspark
findspark.init()

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

AttributeError: module 'socketserver' has no attribute 'UnixStreamServer'

In [4]:
from pyspark.sql.functions import col, explode, array, udf, when, coalesce
# from pyspark.sql import SparkSession
from pyspark.sql.types import StringType


AttributeError: module 'socketserver' has no attribute 'UnixStreamServer'

In [3]:
data = [("Alice", ["Badminton", "Tennis"]), ("Bob", ["Tennis", "Criccket"]), ("Julie", ["Cricket", "Carroms"])]

In [4]:
df = spark.createDataFrame(data,["name","hobbies"])

In [13]:
df.show()

+-----+-------------------+
| name|            hobbies|
+-----+-------------------+
|Alice|[Badminton, Tennis]|
|  Bob| [Tennis, Criccket]|
|Julie| [Cricket, Carroms]|
+-----+-------------------+



In [6]:
df_explode = df.select(col("name"), explode(df.hobbies).alias("hobbies_explode"))
df_explode.show()

+-----+---------------+
| name|hobbies_explode|
+-----+---------------+
|Alice|      Badminton|
|Alice|         Tennis|
|  Bob|         Tennis|
|  Bob|       Criccket|
|Julie|        Cricket|
|Julie|        Carroms|
+-----+---------------+



# FUNCIONES 
- coalesce : Combina multiples columnas 

In [7]:
data_info = [("MD","", "ZR"),("", "ZR", None),(None, "", "BRC")]
schema = "City1 string, City2 string, City3 string"
df_city = spark.createDataFrame(data_info, schema)
df_city.show()

+-----+-----+-----+
|City1|City2|City3|
+-----+-----+-----+
|   MD|     |   ZR|
|     |   ZR| NULL|
| NULL|     |  BRC|
+-----+-----+-----+



In [8]:
df_city = (df_city.withColumn("City1", when(col("City1")=='', None).otherwise(col("City1")))
           .withColumn("City2", when(col("City2")=='', None).otherwise(col("City2")))
           .withColumn("City3", when(col("City3")=='', None).otherwise(col("City3"))))
df_result = df_city.withColumn("Result", coalesce(col("City1"), col("City2"), col("City3"))).select("Result")
df_result.show()

+------+
|Result|
+------+
|    MD|
|    ZR|
|   BRC|
+------+



In [9]:
@udf(returnType=StringType())
def city_value(columnas_city):
    cities = ""
    for city in columnas_city:
        if city is not None and city !="null":
            cities = city
    return cities

In [10]:
lista_cities = [ x for x in df_city.columns if "City" in x ]
print(lista_cities)

['City1', 'City2', 'City3']


In [11]:
df_result = df_city.withColumn("Result", city_value(array(lista_cities)))
df_result.show()

SparkException: Job aborted due to stage failure: Task 0 in stage 1.0 failed 1 times, most recent failure: Lost task 0.0 in stage 1.0 (TID 1) (Rumble executor driver): java.io.IOException: Cannot run program "python3" (in directory "C:\Users\kevin\AppData\Local\Temp\spark-dce76074-69a9-4083-a9f4-b2af2763ff4a\userFiles-1858add3-d007-40b2-bf03-569092333843\f372f30a-7b96-4394-b96d-677093f4fa19"): CreateProcess error=2, El sistema no puede encontrar el archivo especificado
	at java.base/java.lang.ProcessBuilder.start(ProcessBuilder.java:1143)
	at java.base/java.lang.ProcessBuilder.start(ProcessBuilder.java:1073)
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:247)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:154)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:158)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:309)
	at org.apache.spark.sql.execution.